<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c04-header.png" alt="Nextia Learning · SQL and Data Preparation for AI" width="100%">

# Solution: Aggregate correctly

**[Aggregate correctly](https://learning.nextia-ai.com/courses/sql/m02/aggregate-correctly/)** · SQL and Data Preparation for AI · Module 2, lesson 2 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** summarise many rows with `COUNT`, `SUM`, `AVG`, `MIN` and `MAX`, per group with `GROUP BY` and `HAVING`, and see how NULL values, duplicate rows and wrong values change a count or an average without an error.

| | |
|---|---|
| **Time** | About 45 minutes |
| **Needs** | An internet connection for the first cell. No account and no install: only Python's standard library, and pandas to show tables if it is there (it is in Colab and Kaggle). |
| **You should know** | `SELECT`, `WHERE`, `ORDER BY` and `IS NULL`, and that a comparison with NULL is unknown, from [Select and filter](https://learning.nextia-ai.com/courses/sql/m02/select-and-filter/). |
| **You do not need** | The local project from Module 1. The setup below builds the same database here. |
| **Tested** | Python 3.14.6 (SQLite 3.50.4), in Jupyter on macOS, from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/sql/m02/aggregate-correctly/) has the full explanations, the tables of examples and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your query between the triple quotes `"""`, run the cell, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C04/M02-L02-aggregate-correctly/aggregate-correctly-solution.ipynb).

## Setup: get the practice data

Run the next cell. It downloads the three files of the Larkfield help-desk export (about 200 KB), checks each file's checksum, and builds the practice database, `larkfield.db`. It works in a folder `ticket-data`, laid out like your project from Module 1, so the code of the lesson runs here unchanged. It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: customers 240, tickets 1417, outcomes 1874`. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings (it needs a verified phone number).
- `wrong checksum`: delete the folder `ticket-data` (in Colab: the **Files** panel on the left), then run the cell again.

This is the same database that `get_data.py` builds in [Set up a practice database](https://learning.nextia-ai.com/courses/sql/m01/set-up-a-practice-database/#build-the-database). The data is made up for this course; it has no real people.

In [ ]:
# Setup: download the Larkfield export, check it, and build larkfield.db in
# ticket-data/. Standard library only: the same steps as get_data.py.
import csv, hashlib, json, os, sqlite3, subprocess, sys, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C04/"
FILES = {
    "customers.csv": "7b437642f0095524f8311f67a7caeb17100d23eed92618b9cf012cd3d3b79993",
    "tickets.csv": "9c47e08cf2ec5977733d32cc21eb97d0dd01b021d1f5fbf7ec31fef6279b1a28",
    "outcomes.csv": "67b4d9d9275b32061a8888050c351df7314054c11ee1ce29e856c1d5403d0653",
}
# The notebook works in its own folder ticket-data/, marked with a file, so it
# never writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
PROJECT = Path("ticket-data").resolve()
PROJECT.mkdir(exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def download(url, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, expected):
    path = PROJECT / "data" / "raw" / name
    if not (path.exists() and hashlib.sha256(path.read_bytes()).hexdigest() == expected):
        download(LABS + "data/" + name, path)
        if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder ticket-data and run again.")
    return path

DB = PROJECT / "larkfield.db"
if not DB.exists():
    paths = {name: fetch(name, sha) for name, sha in FILES.items()}
    build = sqlite3.connect(DB)
    build.executescript("""
        CREATE TABLE customers (customer_id TEXT, segment TEXT, region TEXT, joined_on TEXT);
        CREATE TABLE tickets (ticket_id TEXT, customer_id TEXT, created_at TEXT, channel TEXT, team TEXT,
            priority INTEGER, order_value REAL, word_count INTEGER, first_reply_minutes INTEGER,
            priority_now INTEGER, closed_at TEXT);
        CREATE TABLE outcomes (outcome_id TEXT, ticket_id TEXT, recorded_at TEXT, status TEXT,
            resolution_code TEXT, csat INTEGER);
        CREATE TABLE snapshot (name TEXT, value TEXT);
        INSERT INTO snapshot VALUES ('source', 'Larkfield help-desk export (synthetic, Nextia Learning C04)'),
            ('exported_at', '2026-07-01 06:00:00'), ('data_version', '1.0');
    """)
    for name, path in paths.items():
        with open(path, newline="", encoding="utf-8") as f:
            rows = list(csv.reader(f))
        marks = ", ".join("?" for _ in rows[0])
        build.executemany(f"INSERT INTO {name[:-4]} ({', '.join(rows[0])}) VALUES ({marks})",
                          [[None if v == "" else v for v in row] for row in rows[1:]])
    build.commit()
    build.close()

os.chdir(PROJECT)  # work in the project folder, as in the lesson
for folder in ("queries", "scripts", "reports"):
    Path(folder).mkdir(exist_ok=True)
# Read-only, as q.py: a query cannot change the data by mistake.
con = sqlite3.connect("file:larkfield.db?mode=ro", uri=True)
counts = [con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0] for t in ("customers", "tickets", "outcomes")]
print("Ready: customers {}, tickets {}, outcomes {}".format(*counts))

## Setup: the helpers

The next cell defines small functions that the lesson uses. You do not need to read them, but you can.

- `sql(query)` runs a query and shows the result as a table, with the number of rows, like `q.py` and the boxes on the lesson page. It shows the first 20 rows; `sql(query, limit=None)` shows all of them.
- `check(query, ...)` compares the result of your query with the expected result, and says what is different. It does not show the answer.
- `expect(what, yours, expected)` compares one of your numbers with the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: show a query result, and check an answer without showing it.
try:
    import pandas as pd
except ImportError:
    pd = None

def run(query, db=None):
    cursor = (db or con).execute(query)
    columns = [d[0] for d in cursor.description] if cursor.description else []
    return columns, cursor.fetchall()

def sql(query, db=None, limit=20):
    """Run a query and show its result as a table, with the row count."""
    columns, rows = run(query, db)
    if not columns:
        print("No result: the cell has no SELECT yet. Write your query between the triple quotes, then run the cell again.")
        return
    shown = rows if limit is None else rows[:limit]
    if pd is not None:
        # Show NULL as NULL, as q.py and the boxes on the lesson page do.
        table = pd.DataFrame(shown, columns=columns, dtype=object)
        display(table.where(table.notna(), "NULL"))
    else:
        print("  ".join(columns))
        for row in shown:
            print("  ".join("NULL" if v is None else str(v) for v in row))
    more = f" (showing {len(shown)}; use limit=None to see all)" if len(shown) < len(rows) else ""
    print(f"{len(rows)} row{'s' if len(rows) != 1 else ''}{more}")

def fingerprint(rows, ordered=False):
    cells = [tuple("NULL" if v is None else f"{v:.6f}" if isinstance(v, float) else str(v) for v in r) for r in rows]
    return hashlib.sha256(repr(cells if ordered else sorted(cells)).encode()).hexdigest()[:16]

def check(query, rows, columns, expected, ordered=False, db=None):
    """Compare your result with the expected one: the column count, the row
    count, then the values (as a fingerprint, so the answer stays hidden)."""
    try:
        got_columns, got = run(query, db)
    except sqlite3.Error as error:
        print(f"Not yet: the query has an error: {error}")
        return
    if len(got_columns) != columns:
        print(f"Not yet: your result has {len(got_columns)} columns; the task asks for {columns}.")
    elif len(got) != rows:
        print(f"Not yet: your result has {len(got)} rows; the task expects {rows}.")
    elif fingerprint(got, ordered) != expected:
        print("Not yet: the number of rows is right, but some values" + (" or their order" if ordered else "") + " are different.")
    else:
        print("Check passed.")

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    same = abs(yours - expected) <= tolerance * max(1, abs(expected)) if isinstance(expected, float) else yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours}, which is not the expected value.")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Six orders, one average

An **aggregation** turns many rows into one value: a count, a total, a mean, a smallest or a largest value. SQL has a function for each: `COUNT`, `SUM`, `AVG`, `MIN` and `MAX`.

Here are six tickets with the value of the order that each one is about, in dollars:

| ticket_id | team | order_value | What the value really is |
|---|---|---|---|
| T-1 | delivery | 40 | an order of $40 |
| T-2 | delivery | 60 | an order of $60 |
| T-3 | returns | 50 | an order of $50 |
| T-4 | account | NULL | no order: an account question |
| T-5 | returns | −1 | "unknown" in the order system |
| T-6 | delivery | 5000 | an order of $50, sent in cents |

The next cell makes this table in memory, as `tiny`. Run the cell. It prints nothing.

In [ ]:
tiny = sqlite3.connect(":memory:")
tiny.executescript("""
CREATE TABLE tickets (ticket_id TEXT, team TEXT, order_value REAL);
INSERT INTO tickets VALUES
    ('T-1', 'delivery', 40),
    ('T-2', 'delivery', 60),
    ('T-3', 'returns',  50),
    ('T-4', 'account',  NULL),
    ('T-5', 'returns',  -1),
    ('T-6', 'delivery', 5000);
""");

The real mean of the four known orders is (40 + 60 + 50 + 50) ÷ 4 = **$50**.

> **Predict.** What does `AVG(order_value)` give? And `COUNT(*)` and `COUNT(order_value)`? Then run the cell.

In [ ]:
sql("""
SELECT
    COUNT(*)           AS all_rows,
    COUNT(order_value) AS with_value,
    SUM(order_value)   AS total,
    AVG(order_value)   AS mean_value,
    MIN(order_value)   AS smallest,
    MAX(order_value)   AS largest
FROM tickets;
""", db=tiny)

> **Check.** You should see `6`, `5`, `5149.0`, `1029.8`, `-1.0` and `5000.0`. Nothing failed: SQL did correct arithmetic on wrong values.

Each row teaches one rule:

- **NULL is left out** of `COUNT(column)`, `SUM`, `AVG`, `MIN` and `MAX`. Here this is right: T-4 has no order, so it must not count as an order of $0.
- **−1 is a sentinel value**: a normal-looking value that stands for something else, here "unknown". SQL adds it like any number.
- **5000 is in the wrong unit.** One value 100 times too large moves the mean much more than all the others.

`MIN` and `MAX` show both problems at once. That is why you print them next to every mean.

Now leave out the sentinel with a filter.

> **Predict.** How many rows does `COUNT(*)` give now? Why is it not 5? Then run the cell.

In [ ]:
sql("""
SELECT COUNT(*) AS all_rows, AVG(order_value) AS mean_value
FROM tickets
WHERE order_value <> -1;
""", db=tiny)

> **Check.** You should see **4**, not 5 (and a mean of 1287.5). The filter drops T-5, and it also drops T-4, because `NULL <> -1` is unknown. This is the rule from the last lesson, inside an aggregation.

## 2. One value per group: GROUP BY

`GROUP BY team` puts the rows with the same team into one group. The aggregations then give one value **per group**, so the result has one row per team.

> **Predict.** How many rows does the result have, and what is the mean of the delivery team? Then run the cell.

In [ ]:
sql("""
SELECT team, COUNT(*) AS ticket_rows, AVG(order_value) AS mean_value
FROM tickets
GROUP BY team
ORDER BY team;
""", db=tiny)

> **Check.** You should see 3 rows: account 1 row with mean NULL (no values to average), delivery 3 rows with mean 1700.0, returns 2 rows with mean 24.5. 1 + 3 + 2 = 6 rows went in.

That is the check for any `GROUP BY`: the counts of the groups add up to the rows that went in. If a grouping column has NULL values, all of them form one group, with NULL as its name.

Every column in `SELECT` must be either in `GROUP BY` or inside an aggregation. `SELECT team, ticket_id … GROUP BY team` asks for one `ticket_id` for a group of three tickets. PostgreSQL stops with an error; SQLite picks one without a warning, so do not write it.

## 3. Three ways to count

From here on, the queries run on the real `tickets` table. `COUNT` has three forms: `COUNT(*)` counts rows, `COUNT(column)` counts the rows where the column is not NULL, and `COUNT(DISTINCT column)` counts the different values, with NULL left out.

> **Predict.** 38 rows have no channel. What is `with_channel`? Then run the cell.

In [ ]:
sql("""
SELECT
    COUNT(*)                    AS all_rows,
    COUNT(order_value)          AS with_order_value,
    COUNT(DISTINCT ticket_id)   AS tickets,
    COUNT(channel)              AS with_channel,
    COUNT(DISTINCT customer_id) AS customers
FROM tickets;
""")

> **Check.** You should see `1417`, `1229`, `1403`, `1379` and `198`.

- `COUNT(*)` counts the 14 duplicate rows from the export twice. `COUNT(DISTINCT ticket_id)` counts each ticket once. "How many tickets?" is **1,403**.
- `COUNT(order_value)` is 188 less than `COUNT(*)`: the 188 account tickets have no order.
- The 12 guest tickets have NULL in `customer_id`, and `COUNT` leaves NULL out, so guests are not "a customer".

## 4. Worked example: the mean order value per team

Priya, the data scientist, plans an `order_value` feature for the model. She asks Mei: "What is a typical order value for each team?" Mei writes one row per team, with the counts next to the mean. `ROUND(x, 2)` rounds to two decimal places. On your computer, this is the file `queries/23-team-summary.sql`.

Run the cell.

In [ ]:
team_summary = """
-- One row per team: rows, distinct tickets and order values.
SELECT
    team,
    COUNT(*)                  AS ticket_rows,
    COUNT(DISTINCT ticket_id) AS tickets,
    COUNT(order_value)        AS with_value,
    ROUND(AVG(order_value), 2) AS avg_value,
    MIN(order_value)          AS min_value,
    MAX(order_value)          AS max_value
FROM tickets
GROUP BY team
ORDER BY team;
"""
sql(team_summary)

> **Check.** You should see 5 rows, from account to warranty. The account row has `0` in `with_value`, and NULL for the mean, the minimum and the maximum. The warranty row has a mean of 535.11 and a maximum of 34518.0.

Mei checks the first two columns: every row must be in exactly one group, and every ticket too. Run the cell. It adds up the two columns of the result.

In [ ]:
columns, rows = run(team_summary)
expect("the sum of ticket_rows", sum(row[1] for row in rows), 1417)
expect("the sum of tickets", sum(row[2] for row in rows), 1403)

> **Check.** You should see `Check passed: the sum of ticket_rows is 1417.` and `Check passed: the sum of tickets is 1403.` The duplicate rows are in four teams.

Then she reads the other columns:

- **With value:** the account team has 0. Grace, the support lead, confirms: "Account questions have no order." Here a missing value has a meaning. It is not "unknown".
- **Min:** −1 in four teams. In the order system, −1 means "unknown". It is a sentinel, not a refund. The table has 11 such rows.
- **Max:** up to 34,518 dollars, for a garden shop.

Mei looks at where the large values come from. The old web form sent cents, not dollars, until February 2026.

> **Predict.** Which channels do the rows above 2,000 come from? Then run the cell.

In [ ]:
sql("""
SELECT channel, COUNT(*) AS rows_above_2000
FROM tickets
WHERE order_value > 2000
GROUP BY channel;
""")

> **Check.** You should see `web_form` 24 and `webform` 1: all 25 rows above 2,000 came from the old web form.

So the mean of the whole column is not a typical order. As a quick check, Mei leaves out the 11 rows of −1 and the 25 rows above 2,000. Run the cell.

In [ ]:
sql("""
SELECT
    (SELECT ROUND(AVG(order_value), 2) FROM tickets) AS mean_all,
    (SELECT ROUND(AVG(order_value), 2) FROM tickets
       WHERE order_value <> -1 AND order_value <= 2000) AS mean_without_sentinels_and_cents;
""")

> **Check.** You should see **299.95** and **123.93**.

That check is not the fix. It drops 25 real orders only because their unit is wrong, and the limit of 2,000 is a guess that works on this export only. Module 4 converts the cents rows and records the sentinel as unknown. For now, Mei writes to Priya: "Do not use `order_value` before cleaning."

> **Your turn: change one thing.** In the team summary cell above, add the line `WHERE order_value <> -1` between `FROM tickets` and `GROUP BY team`. Before you run it, predict the `ticket_rows` of the account team. Then run it, and run the sum cell again.

> **Check.** The account row is gone, and `min_value` is no longer −1. The account team has NULL in every row, and `NULL <> -1` is unknown, so `WHERE` drops all 188 rows: the group does not exist any more. The sum of `ticket_rows` is now 1,218 = 1,417 − 188 − 11, so the sum cell prints `Not yet`. A filter on one column can remove a whole group without a message. Remove the line again and run the cell.

## 5. Conditions on groups: HAVING

`WHERE` filters **rows**, before the grouping. `HAVING` filters **groups**, after the aggregation. So `HAVING` can test an aggregation, and `WHERE` cannot.

Grace asks: "Which teams had at least 40 urgent tickets?"

> **Predict.** How many teams pass? Then run the cell.

In [ ]:
sql("""
SELECT team, COUNT(*) AS urgent_rows
FROM tickets
WHERE priority = 1          -- 1. keep the urgent rows (186 rows)
GROUP BY team               -- 2. make one group per team (5 groups)
HAVING COUNT(*) >= 40       -- 3. keep the groups with 40 or more
ORDER BY urgent_rows DESC;
""")

> **Check.** You should see 2 rows: payment 59 and delivery 49. The other teams have returns 39, warranty 21 and account 18; together 186.

If you write the condition in `WHERE`, SQL stops with an error, because `WHERE` runs before the groups exist. Run the cell to see the error. The cell catches it, so the notebook continues.

In [ ]:
try:
    sql("""
    SELECT team, COUNT(*) AS urgent_rows
    FROM tickets
    WHERE priority = 1 AND COUNT(*) >= 40
    GROUP BY team;
    """)
except sqlite3.Error as error:
    print("SQL error:", error)

> **Check.** You should see `SQL error: misuse of aggregate: COUNT()`.

## 6. Count with a condition: CASE

Sometimes a question needs several counts from the same rows, each with its own condition. `CASE` gives a value that depends on a test:

```sql
CASE WHEN status = 'escalated' THEN ticket_id END
```

This gives `ticket_id` when the status is `escalated`, and NULL in all other rows. `COUNT` leaves NULL out, so `COUNT(DISTINCT CASE WHEN status = 'escalated' THEN ticket_id END)` counts the different tickets that have an escalated event.

On the `outcomes` table, one row is one **outcome event** in a ticket's life.

> **Predict.** Are there more escalated events or more escalated tickets? Then run the cell.

In [ ]:
sql("""
SELECT
    COUNT(*) AS events,
    COUNT(DISTINCT ticket_id) AS tickets_with_events,
    COUNT(CASE WHEN status = 'escalated' THEN outcome_id END) AS escalated_events,
    COUNT(DISTINCT CASE WHEN status = 'escalated' THEN ticket_id END) AS escalated_tickets
FROM outcomes;
""")

> **Check.** You should see `1874` events for `1399` tickets, `228` escalated events and `225` escalated tickets. Three escalation events are logged twice, as you see in the next lesson.

> **Your turn: change one thing.** In the cell above, change both `'escalated'` to `'resolved'`, and run it again. You should see 1533 events and 1396 tickets.

A ticket can have several events, so a count of events is not a count of tickets. For "how many tickets were escalated?", count distinct `ticket_id`.

## 7. Your turn: rows per channel

> **Your turn.** The starting query gives 9 groups, because each spelling is its own group. Change it so that `Email`, `EMAIL` and `email` form one group, and ` chat` joins `chat`. Sort the result from the largest group to the smallest. Then run the check cell. When it is correct, you see 6 rows, the first is `email` with 568 rows, and the check cell prints `Check passed.`

In [ ]:
channels = """
SELECT LOWER(TRIM(channel)) AS channel_for_report, COUNT(*) AS ticket_rows
FROM tickets
GROUP BY LOWER(TRIM(channel))
ORDER BY ticket_rows DESC;
"""
sql(channels)

In [ ]:
check(channels, rows=6, columns=2, expected="ad2481767cd66004", ordered=True)

Six groups remain for four real channels: `webform` and `web_form` are the same channel, and NULL is "not recorded". `LOWER(TRIM())` does not fix a different spelling. Module 4 decides what to do with both.

## 8. Independent variation: one long message

Grace says: "Urgent tickets are long. Customers write more when it is serious." Check her claim with the length of the first message, `word_count`.

> **Your turn.** Write one row per priority, sorted by priority. Give four columns: the priority, the number of rows, the mean `word_count` rounded to 1 decimal place, and the largest `word_count`. Then run the check cell.

In [ ]:
words = """
SELECT
    priority,
    COUNT(*) AS ticket_rows,
    ROUND(AVG(word_count), 1) AS mean_words,
    MAX(word_count) AS max_words
FROM tickets
GROUP BY priority
ORDER BY priority;
"""
sql(words)

In [ ]:
check(words, rows=3, columns=4, expected="f952e822ff20e2f3", ordered=True)

Then decide: is Grace right? Look at the largest value next to each mean.

> **Your turn: change one thing.** In your query, add `WHERE word_count < 10000` before `GROUP BY`, and run the cell again. Compare the mean of priority 1 with the others.

## 9. Failure case: the mean order value includes sentinels and cents

Tomás puts "mean order value: $299.95" in a slide about the model's features. Omar, the product manager, says that a typical Larkfield order is about $100.

**Cause:** `AVG(order_value)` used every non-NULL value: 11 sentinel values of −1, and 25 rows in cents. SQL gave no error, because the arithmetic is correct. The values are wrong.

**Diagnose:** print `COUNT(column)`, `MIN` and `MAX` next to every `AVG`. Run the cell.

In [ ]:
sql("""
SELECT
    COUNT(order_value)         AS with_value,
    ROUND(AVG(order_value), 2) AS mean_value,
    MIN(order_value)           AS smallest,
    MAX(order_value)           AS largest
FROM tickets;
""")

> **Check.** You should see `1229`, `299.95`, `-1.0` and `34518.0`. A minimum of −1 for a price, or a maximum of 34,518 for a garden order, shows the problem at once.

**Fix:** do not report the mean until the column is clean. Module 4 converts the cents rows, marks −1 as unknown, and keeps the original values.

**In AI work:** a model that uses `order_value` learns from the wrong scale. Some preparation steps fill missing values with the mean, or scale values with the mean. They carry the error into every row.

## Check your understanding and recap

Answer the three **knowledge checks** at the end of the [lesson page](https://learning.nextia-ai.com/courses/sql/m02/aggregate-correctly/). They count toward your certificate when you are signed in and enrolled.

In this lesson you learned that `COUNT(*)` counts rows, `COUNT(column)` counts non-NULL values and `COUNT(DISTINCT column)` counts different values; that the aggregations leave out NULL; and that SQL calculates correctly on wrong values, so you print `MIN` and `MAX` next to every mean. `WHERE` filters rows before grouping, `HAVING` filters groups after it, and the group counts must add up to the rows that went in.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Join tables](https://learning.nextia-ai.com/courses/sql/m02/join-tables/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/sql/m02/aggregate-correctly/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.